# Этап 3, версия 1.1. Проверка улучшенной обработки

Проверяем один вопрос: помогли ли микрорайон, флаг неизвестных комнат и более точное заполнение площади? Алгоритм LightGBM, настройки, train и validation остаются одинаковыми. Меняется только обработка данных. Test в этой проверке не используется.

## Одинаковые данные для двух вариантов

Подготовка v1.5 загружает исходный CSV, приводит цены к TRY и воспроизводит разбиение. Старую обученную модель загружаем из файла. Для новой модели все обучаемые шаги — медианы, пороги, One-Hot и скалер — обучаются только на train.

In [1]:
# Выполняем этап 2 v1.5, не дублируя его таблицы и графики.
import json
from pathlib import Path
from IPython.utils.capture import capture_output
stage2=next((p/"notebooks"/"02_preprocessing_v1.5.ipynb"
             for p in (Path.cwd(),Path.cwd().parent)
             if (p/"notebooks"/"02_preprocessing_v1.5.ipynb").exists()),None)
if stage2 is None:
    raise FileNotFoundError("Не найден ноутбук этапа 2 v1.5")
with capture_output():
    for cell in json.loads(stage2.read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"]=="code":
            result=get_ipython().run_cell("".join(cell["source"]))
            if result.error_in_exec:
                raise result.error_in_exec
import cloudpickle
old_path=ROOT/"models"/"price_model_v1.0.pkl"
with old_path.open("rb") as f:
    old_bundle=cloudpickle.load(f)
assert old_bundle["model_name"]=="LightGBM"
print("Train:",len(train),"Validation:",len(val),"Test не открываем")
print("Старая модель:",old_path)

Train: 200662 Validation: 42720 Test не открываем
Старая модель: C:\ForesightEstate\models\price_model_v1.0.pkl


Размеры частей сохранились: новая обработка не удаляет строки. Сравнение будет парным — оба прогноза относятся к одним и тем же объявлениям validation.

## Новая LightGBM с теми же настройками

Иерархия заполнения площади следует этапу 2: микрорайон+тип → район+тип → тип → общая медиана train. Редкие микрорайоны объединяет One-Hot. Числовые признаки стандартизируются так же, как в сохранённой модели.

In [2]:
import time
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_absolute_percentage_error,r2_score,median_absolute_error
from lightgbm import LGBMRegressor

class AreaByLocation(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        def level(keys):
            t=X.groupby(keys).size_m2.agg(["median","count"])
            return t.loc[t["count"].ge(20),"median"]
        self.neighborhood_=level(["neighborhood","sub_type"])
        self.district_=level(["district","sub_type"])
        self.type_=X.groupby("sub_type").size_m2.median()
        self.global_=X.size_m2.median()
        return self
    def transform(self,X):
        out=X.copy()
        for keys,table in [(["neighborhood","sub_type"],self.neighborhood_),
                           (["district","sub_type"],self.district_)]:
            idx=pd.MultiIndex.from_frame(out[keys])
            values=pd.Series(table.reindex(idx).to_numpy(),index=out.index)
            out["size_m2"]=out.size_m2.fillna(values)
        out["size_m2"]=out.size_m2.fillna(out.sub_type.map(self.type_)).fillna(self.global_)
        return out

class AreaCap(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        self.low_,self.high_=X.size_m2.quantile([.001,.999])
        return self
    def transform(self,X):
        out=X.copy()
        out["size_m2"]=out.size_m2.clip(self.low_,self.high_)
        return out

numeric=Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())])
categorical=Pipeline([("fill",SimpleImputer(strategy="most_frequent")),
                      ("onehot",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=50))])
encoder=ColumnTransformer([("num",numeric,num),("cat",categorical,cat)],sparse_threshold=.8)
regressor=TransformedTargetRegressor(
    regressor=LGBMRegressor(random_state=42,verbosity=-1,n_jobs=4,num_leaves=15),
    func=np.log1p,inverse_func=np.expm1)
new_model=Pipeline([("area",AreaByLocation()),("cap",AreaCap()),
                    ("encode",encoder),("model",regressor)])
X_train_new=features(train)
X_val_new=features(val)
start=time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    new_model.fit(X_train_new,train.price)
fit_seconds=time.perf_counter()-start
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    new_pred=new_model.predict(X_val_new)
    old_pred=old_bundle["pipeline"].predict(old_bundle["feature_builder"](val))
print("Новая модель обучена за",round(fit_seconds,2),"с")
print("Новые признаки: микрорайон и rooms_missing; число столбцов после кодирования:",
      new_model.named_steps["encode"].transform(new_model.named_steps["cap"].transform(
          new_model.named_steps["area"].transform(X_val_new.head(1)))).shape[1])

Новая модель обучена за 3.91 с
Новые признаки: микрорайон и rooms_missing; число столбцов после кодирования: 1259


Алгоритм и его основные настройки не изменились. Поэтому разница в ошибках ниже относится прежде всего к новым признакам и заполнению площади, а не к подбору другой модели.

## Сравнение на всей validation

MAE и медианная абсолютная ошибка выражены в TRY. Меньше — лучше. Дополнительно показываем RMSE, MAPE и R²; огромные цены могут заметно менять RMSE и R².

In [3]:
def metrics(name,y,p):
    return {"Вариант":name,"MAE, TRY":mean_absolute_error(y,p),
            "RMSE, TRY":np.sqrt(mean_squared_error(y,p)),
            "MAPE, %":100*mean_absolute_percentage_error(y,p),
            "R²":r2_score(y,p),"MedAE, TRY":median_absolute_error(y,p)}
quality=pd.DataFrame([metrics("v1.4 — прежняя",val.price,old_pred),
                      metrics("v1.5 — новая",val.price,new_pred)])
display(quality.round(2))
gain=quality.loc[0,"MAE, TRY"]-quality.loc[1,"MAE, TRY"]
print("Снижение MAE (плюс = новая лучше), TRY:",round(gain,2))
assert len(old_pred)==len(new_pred)==len(val)

,Вариант,"MAE, TRY","RMSE, TRY","MAPE, %",R²,"MedAE, TRY"
0,v1.4 — прежняя,310847.20,12663562.51,102.61,0.02,55561.59
1,v1.5 — новая,309375.97,12670752.32,103.88,0.02,55092.64


Снижение MAE (плюс = новая лучше), TRY: 1471.23


На всей validation новая обработка снизила MAE с 310 847 до 309 376 TRY — выигрыш 1 471 TRY (около 0,47%). Медианная ошибка уменьшилась с 55 562 до 55 093 TRY, но RMSE и MAPE немного выросли. R² при округлении до двух знаков остался около 0,02. Одного небольшого улучшения MAE недостаточно, чтобы объявить новую модель лучше прежней.


In [4]:
# Все группы заданы до просмотра ошибок.
report=pd.DataFrame({"Тип жилья":val.sub_type.to_numpy(),
                     "Цена":val.price.to_numpy(),
                     "Ошибка v1.4":np.abs(val.price.to_numpy()-old_pred),
                     "Ошибка v1.5":np.abs(val.price.to_numpy()-new_pred)})
types=report.groupby("Тип жилья").agg(
    Объявлений=("Цена","size"),MAE_v14=("Ошибка v1.4","mean"),MAE_v15=("Ошибка v1.5","mean"))
types["Изменение, TRY"]=types.MAE_v14-types.MAE_v15
display(types.sort_values("Объявлений",ascending=False).head(8).round(1))
report["Диапазон"]=pd.cut(report["Цена"],[0,100_000,300_000,1_000_000,10_000_000,np.inf],
                          labels=["до 100 тыс.","100–300 тыс.","300 тыс.–1 млн",
                                  "1–10 млн","свыше 10 млн"])
bands=report.groupby("Диапазон",observed=True).agg(
    Объявлений=("Цена","size"),MAE_v14=("Ошибка v1.4","mean"),MAE_v15=("Ошибка v1.5","mean"))
bands["Изменение, TRY"]=bands.MAE_v14-bands.MAE_v15
display(bands.round(1))

,Объявлений,MAE_v14,MAE_v15,"Изменение, TRY"
Тип жилья,,,,
Daire,36593,173542.8,171218.1,2324.7
Villa,2812,934478.9,944315.1,-9836.2
Müstakil Ev,1213,361354.0,358151.1,3202.9
Yazlık,853,161083.9,161705.4,-621.5
Rezidans,620,1003731.8,1020219.2,-16487.4
Komple Bina,403,1413336.6,1418242.6,-4906.0
Prefabrik Ev,98,33706.5,35083.4,-1376.9
Köşk / Konak / Yalı,54,42647176.7,42421630.8,225545.9


,Объявлений,MAE_v14,MAE_v15,"Изменение, TRY"
Диапазон,,,,
до 100 тыс.,1391,99924.6,100136.4,-211.8
100–300 тыс.,22864,48676.3,47876.8,799.5
300 тыс.–1 млн,15869,143322.8,140243.9,3078.9
1–10 млн,2428,1112805.3,1113688.8,-883.5
свыше 10 млн,168,41971318.9,41982319.2,-11000.3


У квартир, составляющих большую часть validation, MAE уменьшилась примерно на 2 325 TRY; для вилл она выросла примерно на 9 836 TRY, для резиденций — на 16 487 TRY. В диапазонах 100–300 тыс. и 300 тыс.–1 млн TRY новая обработка помогает, но у самых дешёвых и дорогих объектов выигрыша нет. Это объясняет, почему общий эффект небольшой.


## Устойчивость разницы

Посчитаем приближённый 95%-й интервал для снижения MAE с повторной выборкой целых групп похожих объявлений. Так одинаковые объекты остаются вместе. Если интервал охватывает ноль, улучшение на этой validation нельзя считать убедительным.

In [5]:
group=pd.util.hash_pandas_object(
    val[["address","sub_type","room_count","size"]],index=False).to_numpy()
delta=report["Ошибка v1.4"].to_numpy()-report["Ошибка v1.5"].to_numpy()
grouped=pd.DataFrame({"group":group,"delta":delta}).groupby("group").delta.agg(["sum","size"])
sums=grouped["sum"].to_numpy(); counts=grouped["size"].to_numpy()
rng=np.random.default_rng(42)
boot=[]
for _ in range(500):
    ids=rng.integers(0,len(grouped),size=len(grouped))
    boot.append(sums[ids].sum()/counts[ids].sum())
ci=np.quantile(boot,[.025,.975])
print("Снижение MAE, TRY:",round(delta.mean(),2))
print("95%-й групповой bootstrap-интервал:",[float(round(x,2)) for x in ci])
print("Убедительное улучшение по этому критерию:",bool(ci[0]>0))

Снижение MAE, TRY: 1471.23
95%-й групповой bootstrap-интервал: [-225.09, 3309.64]
Убедительное улучшение по этому критерию: False


Групповой 95%-й bootstrap-интервал снижения MAE составляет примерно от −225 до +3 310 TRY. Ноль лежит внутри интервала: на этой validation нет убедительного доказательства, что новая обработка стабильно лучше прежней. Доверительный интервал не является гарантией качества на будущих объектах.


## Решение и сохранение кандидата

Новая модель будет сохранена отдельно только при положительном снижении MAE и интервале целиком выше нуля. Прежняя модель остаётся без изменений. Даже сохранённый кандидат не считается окончательно проверенным: для будущей версии приложения понадобится новый независимый набор.

In [6]:
candidate=ROOT/"models"/"price_model_v1.1_candidate.pkl"
confirmed=(gain>0) and (ci[0]>0)
if confirmed:
    bundle={"feature_builder":features,"pipeline":new_model,
            "input_columns":old_bundle["input_columns"],
            "target_currency":"TRY","model_name":"LightGBM",
            "preprocessing_version":"1.5"}
    with candidate.open("wb") as f:
        cloudpickle.dump(bundle,f)
    with candidate.open("rb") as f:
        restored=cloudpickle.load(f)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        check=restored["pipeline"].predict(restored["feature_builder"](val.head(3)))
    assert np.allclose(check,new_pred[:3])
    print("Кандидат сохранён и проверен:",candidate)
else:
    print("Убедительного улучшения нет; новую модель не сохраняем.")
print("Test не использован.")

Убедительного улучшения нет; новую модель не сохраняем.
Test не использован.


## Заключение

Микрорайон и более локальное заполнение площади дали небольшое снижение MAE на общей validation, но ухудшили некоторые типы жилья; групповой интервал включает ноль. Поэтому прежняя модель приложения остаётся основной, кандидат v1.5 не сохраняется, а test не используется для настройки. Следующий разумный эксперимент — отдельно проверить модель квартир и исследовать подозрительные значения площади и цены на train, не исключая записи ради красивой метрики.
